# Clean, count and categorize paper text

Clean text copied from any journal article and count paragraphs, words and characters.

1. Paste your text into **Original text** and click **Find line-end hyphens**.
2. Select any splits that should keep a genuine hyphen, then click **Apply cleanup**.
3. Review and edit **Cleaned text**. Paragraph, word and character counts update as you type.
4. Use **Copy paragraph count** and **Copy word count** to copy plain digits without dots, commas or labels.

The default paragraph rule counts body lines ending in `.`, `:`, `?` or `!`, including trailing footnote numbers directly attached to the punctuation (for example, `.12` or `?3`). Numbers starting with zero are excluded from this footnote rule, so a line ending in `p = 0.001` continues into the next line. P-value expressions without a leading zero, such as `(𝑝<.1`, also continue; a wrapped closing parenthesis attaches without an extra space. Cleanup separates paragraphs and numbered headings with an empty line, preserving existing blank lines. Other body breaks are replaced with spaces, including intervening blank lines. Single-character lines attach to the body paragraph above, without crossing numbered headings. Numbered headings such as `1. Introduction` and `2.1. Methods` remain separate and are excluded from paragraph counts. These automatic rules are heuristics; compare the resulting boundaries with the PDF.

The alternative blank-line mode is still available. Line unwrapping is optional in that mode. Word and character counts cover the displayed text, including headings and any retained mathematical text. Contractions and internally hyphenated compounds count as one word. There is no formula-exclusion or manual break-review workflow in this version.

Clipboard buttons work on Windows and macOS when this notebook runs on your computer. Processing runs locally without AI. Applying cleanup again replaces your edited result; copy your text before restarting the kernel or creating another result.


Run all cells to open two independent tabs: **Clean & count (original)** retains the existing workflow; **Categorize full paper** adds a saved, editable section categorization. The two tabs do not share text or selections.


In [ ]:
# Run this cell if the widget cell reports that ipywidgets is missing.
# Uncomment the following line, run the cell, then restart the kernel if needed:
# %pip install ipywidgets


In [ ]:
import re
from html import escape
from unicodedata import normalize

LINE_END_HYPHEN = re.compile(r"(?<=\w)[-\u2010\u00ad][ \t]*\n[ \t]*(?=\w)")
# Numbered section/subsection headings are excluded from paragraph counts.
NUMBERED_HEADING = re.compile(r"^[^\S\n]*(?:\d+\.|\d+(?:\.\d+)+\.?)[^\S\n]+[^\W\d_].*$")
WORD = re.compile(r"[^\W_]+(?:['’\-\u2010\u2011][^\W_]+)*", re.UNICODE)


def normalize_newlines(text):
    return text.replace("\r\n", "\n").replace("\r", "\n")


def find_line_end_hyphens(text):
    """Return numbered split occurrences for the review selector."""
    text = normalize_newlines(text)
    splits = []
    for index, match in enumerate(LINE_END_HYPHEN.finditer(text)):
        left = re.search(r"[\w'’\-\u2010\u2011]+$", text[:match.start()])
        right = re.match(r"[\w'’\-\u2010\u2011]+", text[match.end():])
        left, right = left.group(0), right.group(0)
        splits.append({
            "index": index,
            "line": text.count("\n", 0, match.start()) + 1,
            "left": left,
            "right": right,
            "joined": left + right,
            "hyphenated": left + "-" + right,
        })
    return splits



def ends_paragraph(text):
    """Recognize final punctuation or footnote digits without a leading zero."""
    text = text.rstrip()
    # Normalize only for recognition: mathematical italic 𝑝 also means p.
    # A p-value can omit the leading zero, e.g. (𝑝<.1 followed by a wrap.
    if re.search(r"(?<!\w)p\s*(?:[<>=≤≥≈]\s*)+[+-]?(?:\d+)?\.\d+$",
                 normalize("NFKC", text), re.IGNORECASE):
        return False
    # A zero-prefixed suffix is likely a decimal, such as p = 0.001.
    return re.search(r"[.:?!](?:[1-9]\d*)?$", text) is not None


def attach_single_character_lines(text):
    """Attach equation remnants to the preceding body line, never across a heading."""
    output = []
    previous_body = None
    for line in normalize_newlines(text).split("\n"):
        stripped = line.strip()
        if NUMBERED_HEADING.fullmatch(line):
            output.append(line)
            previous_body = None
        elif len(stripped) == 1:
            if previous_body is not None:
                output[previous_body] = output[previous_body].rstrip() + " " + stripped
            else:
                output.append(line)
        else:
            output.append(line)
            if stripped:
                previous_body = len(output) - 1
    return "\n".join(output)


def join_unfinished_lines(text):
    """Join unfinished body lines and ensure blank lines between paragraphs."""
    parts = re.split(r"(\n(?:[^\S\n]*\n)*)", attach_single_character_lines(text))
    for index in range(1, len(parts), 2):
        previous, following = parts[index - 1], parts[index + 1]
        if (previous.strip() and following.strip()
                and not NUMBERED_HEADING.fullmatch(previous)
                and not NUMBERED_HEADING.fullmatch(following)
                and not ends_paragraph(previous)):
            parts[index - 1] = previous.rstrip()
            # Wrapped closing brackets attach directly to the expression.
            parts[index] = "" if following.lstrip().startswith((")", "]", "}")) else " "
            parts[index + 1] = following.lstrip()
        elif previous.strip() and following.strip() and parts[index].count("\n") == 1:
            # Preserve existing blank lines; add one only when it is missing.
            parts[index] = "\n\n"
    return "".join(parts)


def clean_text(text, keep_hyphens=(), unwrap_lines=False):
    """Join split words; retain a genuine hyphen for selected occurrence indexes."""
    text = normalize_newlines(text)
    keep_hyphens = set(keep_hyphens)
    matches = iter(enumerate(LINE_END_HYPHEN.finditer(text)))

    def replacement(match):
        index, _ = next(matches)
        if index not in keep_hyphens:
            return ""
        # A soft hyphen is invisible; replace it with a visible real hyphen.
        return "-" if match.group(0)[0] == "\u00ad" else match.group(0)[0]

    cleaned = join_unfinished_lines(LINE_END_HYPHEN.sub(replacement, text))
    if unwrap_lines:
        cleaned = re.sub(r"[ \t]*\n[ \t]*", "\n", cleaned)
        lines = cleaned.split("\n")
        pieces = []
        for index, line in enumerate(lines):
            pieces.append(line)
            if index + 1 < len(lines):
                following = lines[index + 1]
                join = (line.strip() and following.strip()
                        and not NUMBERED_HEADING.fullmatch(line)
                        and not NUMBERED_HEADING.fullmatch(following))
                pieces.append(" " if join else "\n")
        cleaned = "".join(pieces)
    return cleaned


def count_paragraphs(text, paragraph_mode="lines"):
    lines = join_unfinished_lines(text).split("\n")
    if paragraph_mode == "lines":
        return sum(len(line.strip()) > 1 and ends_paragraph(line)
                   and not NUMBERED_HEADING.fullmatch(line) for line in lines)
    if paragraph_mode != "blank_lines":
        raise ValueError("paragraph_mode must be 'lines' or 'blank_lines'")
    paragraphs = 0
    last_body_line = ""
    for line in lines + [""]:
        if not line.strip() or NUMBERED_HEADING.fullmatch(line):
            paragraphs += int(ends_paragraph(last_body_line))
            last_body_line = ""
        elif len(line.strip()) > 1:
            last_body_line = line
    return paragraphs


def count_text(text, paragraph_mode="lines"):
    return {
        "paragraphs": count_paragraphs(text, paragraph_mode),
        "words": len(WORD.findall(text)),
        "characters_including_whitespace": len(text),
        "characters_excluding_whitespace": sum(not char.isspace() for char in text),
    }


In [ ]:
# Checks for automatic paragraph cleanup and counting.
assert clean_text("exam-\nple.") == "example."
assert clean_text("well-\nknown.", keep_hyphens={0}) == "well-known."
assert clean_text("We define\n\nx = y\nand continue.") == "We define x = y and continue."
assert clean_text("Body.\n.\nNext.") == "Body. .\n\nNext."
assert count_text("1. Introduction\nFirst.\nSecond:")["paragraphs"] == 2
assert count_text(".")["paragraphs"] == 0
assert clean_text("Fragment\n2.1. Methods\nBody.") == "Fragment\n\n2.1. Methods\n\nBody."
# Every supported ending separates paragraphs in both counting modes.
for ending in (".", ":", "?", "!", ".1", ".12", ":2", "?3", "!45", "1."):
    original = f"First{ending}\nNext."
    expected = f"First{ending}\n\nNext."
    assert clean_text(original) == expected
    assert clean_text(original, unwrap_lines=True) == expected
    for mode in ("lines", "blank_lines"):
        assert count_text(original, paragraph_mode=mode)["paragraphs"] == 2
        assert count_text(expected, paragraph_mode=mode)["paragraphs"] == 2
    assert clean_text(expected) == expected  # Cleanup is stable when repeated.
for separator in ("\n\n", "\n\n\n", "\n \t\n"):
    original = "First?" + separator + "Next!"
    assert clean_text(original) == original
assert clean_text("First?\r\nNext!") == "First?\n\nNext!"
assert clean_text("A wrapped\nline?\nNext!") == "A wrapped line?\n\nNext!"
assert clean_text("A value of 12\ncontinues.") == "A value of 12 continues."
assert not ends_paragraph("Unfinished 12")
assert not ends_paragraph("Text. 12")  # Footnote digits must touch the punctuation.
# Zero-prefixed suffixes are decimals, not footnote paragraph endings.
for suffix in ("0", "00", "001", "01", "0001"):
    first_line = f"The result has p = 0.{suffix}"
    assert not ends_paragraph(first_line)
    for separator in ("\n", "\n\n"):
        original = first_line + separator + "and remains significant."
        expected = first_line + " and remains significant."
        assert clean_text(original) == expected
        for mode in ("lines", "blank_lines"):
            assert count_text(original, paragraph_mode=mode)["paragraphs"] == 1
# Actual sentence punctuation after a decimal still ends the paragraph.
assert clean_text("The result has p = 0.001.\nNext.") == "The result has p = 0.001.\n\nNext."
# Decimal p-values without a leading zero must not look like footnotes.
for expression in ("𝑝<.1", "p < .1", "p = .25", "𝑝 ≤ .1", "P >= .2", "p < 0.1"):
    assert not ends_paragraph("Result (" + expression)
    for separator in ("\n", "\n\n", "\r\n"):
        original = "(" + expression + separator + ")."
        expected = "(" + expression + ")."
        assert clean_text(original) == expected
        assert clean_text(original, unwrap_lines=True) == expected
        assert clean_text(expected) == expected
        for mode in ("lines", "blank_lines"):
            assert count_text(original, paragraph_mode=mode)["paragraphs"] == 1
            assert count_text(original + "\nNext.", paragraph_mode=mode)["paragraphs"] == 2
        assert clean_text(original + "\nNext.") == expected + "\n\nNext."
assert clean_text("Significant (𝑝<.1\n), with controls.") == "Significant (𝑝<.1), with controls."
assert clean_text("Result p < .1\nand other evidence.") == "Result p < .1 and other evidence."
print("Paragraph cleanup checks passed. Run all cells to open both workflows.")


In [ ]:
import shutil
import subprocess
import sys

try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError as error:
    raise ImportError("Install ipywidgets using the setup cell above, then run this cell again.") from error

original_editor = widgets.Textarea(
    description="Original text:",
    placeholder="Paste text copied from your PDF here…",
    layout=widgets.Layout(width="100%", height="240px"),
    style={"description_width": "110px"},
)
cleaned_editor = widgets.Textarea(
    description="Cleaned text:",
    placeholder="The cleaned text will appear here. You can edit it directly.",
    layout=widgets.Layout(width="100%", height="300px"),
    style={"description_width": "110px"},
)
keep_selector = widgets.SelectMultiple(
    options=[],
    description="Keep hyphens:",
    rows=6,
    layout=widgets.Layout(width="100%"),
    style={"description_width": "110px"},
)
paragraph_mode_selector = widgets.Dropdown(
    options=[("Body lines ending in . : ? or ! (optional footnote digits without leading zeros)", "lines"),
             ("Blocks separated by blank lines (wrapped text)", "blank_lines")],
    value="lines", description="Paragraph rule:",
    layout=widgets.Layout(width="100%"), style={"description_width": "110px"},
)
paragraph_rule_help = widgets.HTML()
unwrap_checkbox = widgets.Checkbox(
    value=False, disabled=True, description="Unwrap body lines (blank-line mode only)", indent=False,
    layout=widgets.Layout(width="100%"),
)
find_button = widgets.Button(description="Find line-end hyphens", layout=widgets.Layout(width="200px"))
apply_button = widgets.Button(description="Apply cleanup", button_style="primary")
clear_original_button = widgets.Button(
    description="Clear original text", button_style="danger",
    layout=widgets.Layout(width="180px", margin="0 0 0 32px", flex="0 0 auto"),
)
status = widgets.HTML()
counts_display = widgets.HTML()
headings_display = widgets.HTML()
copy_paragraph_button = widgets.Button(description="Copy paragraph count", layout=widgets.Layout(width="200px"))
copy_word_button = widgets.Button(description="Copy word count", layout=widgets.Layout(width="180px"))
copy_status = widgets.HTML()
review_state = {"source": None}


def update_counts(change=None):
    counts = count_text(cleaned_editor.value, paragraph_mode=paragraph_mode_selector.value)
    copy_status.value = ""
    counts_display.value = (
        f"<p><strong>Paragraphs:</strong> {counts['paragraphs']}<br>"
        f"<strong>Words:</strong> {counts['words']}<br>"
        f"<strong>Characters (including whitespace):</strong> "
        f"{counts['characters_including_whitespace']}<br>"
        f"<strong>Characters (excluding whitespace):</strong> "
        f"{counts['characters_excluding_whitespace']}</p>"
    )

    headings = [line.strip() for line in normalize_newlines(cleaned_editor.value).split("\n")
                if NUMBERED_HEADING.fullmatch(line)]
    headings_display.value = (
        f"<p><strong>Numbered headings excluded from paragraph count ({len(headings)}):</strong></p>"
        + "<ul>" + "".join(f"<li>{escape(heading)}</li>" for heading in headings) + "</ul>"
        if headings else ""
    )


def copy_count(metric):
    value = str(count_text(cleaned_editor.value, paragraph_mode=paragraph_mode_selector.value)[metric])
    utility = {"darwin": "pbcopy", "win32": "clip.exe"}.get(sys.platform)
    command = shutil.which(utility) if utility else None
    if not command:
        copy_status.value = "<p>Could not access the clipboard. Select the displayed number and copy it manually.</p>"
        return
    try:
        encoding = "utf-16-le" if sys.platform == "win32" else "utf-8"
        options = {"creationflags": subprocess.CREATE_NO_WINDOW} if sys.platform == "win32" else {}
        subprocess.run([command], input=value.encode(encoding), capture_output=True, check=True, timeout=5, **options)
    except (OSError, subprocess.SubprocessError):
        copy_status.value = "<p>Could not access the clipboard. Select the displayed number and copy it manually.</p>"
        return
    label = "paragraph count" if metric == "paragraphs" else "word count"
    copy_status.value = f"<p>Copied {label}: <strong>{value}</strong></p>"


def find_splits(button=None):
    splits = find_line_end_hyphens(original_editor.value)
    keep_selector.options = [
        (f"{split['index'] + 1}. Line {split['line']}: "
         f"{split['left']}- | {split['right']} → {split['joined']} "
         f"(select to keep {split['hyphenated']})", split['index'])
        for split in splits
    ]
    keep_selector.value = ()
    review_state["source"] = original_editor.value
    status.value = (
        f"<p>Found <strong>{len(splits)}</strong> split(s). "
        "Select any that need a real hyphen, then click <strong>Apply cleanup</strong>.</p>"
    )


def apply_cleanup(button=None):
    # Selected indexes must refer to the exact original text the user reviewed.
    if review_state["source"] != original_editor.value:
        find_splits()
        if keep_selector.options:
            status.value += "<p>The original text changed. Review the refreshed list and click Apply cleanup again.</p>"
            return
    cleaned_editor.value = clean_text(
        original_editor.value, keep_hyphens=keep_selector.value,
        unwrap_lines=unwrap_checkbox.value and paragraph_mode_selector.value == "blank_lines",
    )
    update_counts()
    status.value = (
        f"<p>Cleanup applied; kept <strong>{len(keep_selector.value)}</strong> selected hyphen(s). "
        "Review and edit the cleaned text below. Counts use the current edited text.</p>"
    )


def clear_original_text(button=None):
    review_state["source"] = None
    original_editor.value = ""
    keep_selector.options = []
    keep_selector.value = ()
    status.value = "<p>Original text cleared.</p>"


def original_changed(change):
    if review_state["source"] is not None and original_editor.value != review_state["source"]:
        status.value = "<p>The original text changed. Click <strong>Find line-end hyphens</strong> to refresh the review list.</p>"


def paragraph_mode_changed(change=None):
    line_based = paragraph_mode_selector.value == "lines"
    unwrap_checkbox.disabled = line_based
    if line_based:
        unwrap_checkbox.value = False
    paragraph_rule_help.value = (
        "<p>Body lines ending in . : ? or ! (optional footnote digits without leading zeros) count as paragraphs. Cleanup adds missing blank lines between paragraphs and joins unfinished body breaks with a space. Numbered headings are excluded.</p>"
        if line_based else
        "<p>Blocks ending in . : ? or ! (optional footnote digits without leading zeros) count as paragraphs. Cleanup adds missing blank lines between paragraphs and joins breaks after unfinished body lines, including blank lines. Numbered headings are excluded.</p>"
    )
    update_counts()


copy_paragraph_button.on_click(lambda button: copy_count("paragraphs"))
copy_word_button.on_click(lambda button: copy_count("words"))
find_button.on_click(find_splits)
apply_button.on_click(apply_cleanup)
clear_original_button.on_click(clear_original_text)
original_editor.observe(original_changed, names="value")
cleaned_editor.observe(update_counts, names="value")
paragraph_mode_selector.observe(paragraph_mode_changed, names="value")
paragraph_mode_changed()
legacy_workflow = widgets.VBox([
    original_editor,
    widgets.HBox([
        widgets.HBox([find_button, apply_button]),
        clear_original_button,
    ], layout=widgets.Layout(width="100%", justify_content="space-between", align_items="center")),
    paragraph_mode_selector,
    paragraph_rule_help,
    unwrap_checkbox,
    status,
    keep_selector,
    widgets.HTML("<p><strong>Review the cleaned text:</strong> You can restore or remove hyphens and make any other edits here.</p>"),
    cleaned_editor,
    counts_display,
    widgets.HBox([copy_word_button, copy_paragraph_button]),
    copy_status,
    headings_display,
])


In the **Clean & count (original)** tab, counts reflect **Cleaned text**. In another cell, run
`count_text(cleaned_editor.value, paragraph_mode=paragraph_mode_selector.value)`.

The full-paper tab has its own input, cleaned text, category assignments and save/reload controls.

## Full-paper categorization

Paste the full paper's prose, with headings, excluding table text, figure notes and formulas. Review line-end hyphens and clean the paper, then review the suggested heading boundaries. The nine categories are **Introduction, Literature Review, Theory, Data, Methods, Results, Discussion, Conclusion, Not categorizable**. Unassigned text is reported; excluded parts remain in the saved text and JSON.

Each section/subsection owns only the text up to the next heading. Parent titles provide context in the summary; child text is never counted twice. Heading words are included by default to preserve the original word-count convention; paragraph counts use body text and the existing punctuation/footnote rules. You may turn heading words off.

The editable outline supports levels 1–6, including unnumbered headings. Detection and optional label suggestions are local heuristics; review them against the paper. Save creates `<prefix>_cleaned.txt` and `<prefix>_categories.json`. Load the JSON to revise labels later, with counts recomputed from the saved text. Existing files are protected unless overwrite is enabled. The notebook itself does not save your paper automatically.

In **Save and reload**, a generated journal project provides its sampled-paper list and all saved categorization JSON files. Every save in that project goes into `paper_categorizations/`. Choose a sampled paper to start/reopen it, or associate text already pasted. Use **Show text** / **Hide text** to open and completely close the floating section preview.


The summary warns only about **Unassigned** sections and highlights their rows above. Empty categories are allowed. **Not categorizable** is a normal counted category; **Exclude from summary** deliberately omits a part. Individual title buttons copy one title per line.


In [ ]:
# Full-paper categorization: self-contained, local helpers.
import json
import csv
import hashlib
import os
from collections import defaultdict, deque
from datetime import datetime, timezone
from pathlib import Path
import tempfile

PAPER_CATEGORIES = (
    "Introduction", "Literature Review", "Theory", "Data", "Methods",
    "Results", "Discussion", "Conclusion", "Not categorizable",
)
PAPER_UNASSIGNED = "Unassigned"
PAPER_EXCLUDED = "Exclude from summary"
PAPER_LABELS = (PAPER_UNASSIGNED, *PAPER_CATEGORIES, PAPER_EXCLUDED)
PAPER_NUMBERED_TITLE = re.compile(r"^(\d+(?:\.\d+)*)(?:\.)?\s+([^\W\d_].*)$", re.UNICODE)
PAPER_KNOWN_TITLES = {
    "abstract", "introduction", "literature review", "related literature", "related work",
    "background", "institutional background", "institutional setting", "theory",
    "theoretical framework", "conceptual framework", "data", "data description",
    "data and methods", "data and empirical strategy", "methods", "methodology",
    "empirical methods", "empirical strategy", "research design", "identification",
    "identification strategy", "estimation", "empirical model", "results", "empirical results",
    "robustness checks", "sensitivity analysis", "discussion", "results and discussion",
    "discussion and conclusion", "conclusion", "conclusions", "concluding remarks",
    "references", "bibliography", "acknowledgments", "acknowledgements", "appendix",
}


def paper_detect_headings(text):
    """Suggest short numbered/unnumbered heading lines; the outline remains editable."""
    lines = normalize_newlines(text).split("\n")
    headings = []
    small_words = {"a", "an", "and", "as", "at", "by", "for", "from", "in", "of", "on", "or", "the", "to", "with"}
    for index, raw in enumerate(lines):
        title = raw.strip()
        words = WORD.findall(title)
        if not title or len(title) > 140 or not words or len(words) > 18:
            continue
        numbered = PAPER_NUMBERED_TITLE.fullmatch(title)
        if numbered:
            parts = numbered.group(1).split(".")
            if len(parts) <= 6 and all(int(number) < 100 for number in parts):
                headings.append({"title": title, "level": len(parts), "line": index + 1})
            continue
        if title.endswith((".", "?", "!", ";", ",")) or re.search(r"[.?!]\d+$", title):
            continue
        name = " ".join(title.rstrip(":").casefold().split())
        known = name in PAPER_KNOWN_TITLES or bool(re.fullmatch(r"appendix\s+[a-z](?:[.:].*)?", name))
        if ends_paragraph(title) and not known:
            continue
        before_blank = index == 0 or not lines[index - 1].strip()
        after_blank = index + 1 == len(lines) or not lines[index + 1].strip()
        content_words = [word for word in words if word.casefold() not in small_words]
        title_case = bool(content_words) and sum(word[0].isupper() for word in content_words) / len(content_words) >= 0.75
        standalone = before_blank and after_blank and len(words) <= 12 and title[0].isupper()
        if known or (len(words) <= 12 and title_case and before_blank) or standalone:
            headings.append({"title": title, "level": 1, "line": index + 1})
    return headings


def paper_clean_text(text, keep_hyphens=()):
    """Apply the existing rules within body spans, preserving suggested heading lines."""
    text = normalize_newlines(text)
    keep = set(keep_hyphens)
    index = 0

    def replace_hyphen(match):
        nonlocal index
        preserve = index in keep
        index += 1
        if not preserve:
            return ""
        return "-" if match.group(0)[0] == "\u00ad" else match.group(0)[0]

    text = LINE_END_HYPHEN.sub(replace_hyphen, text)
    lines = text.split("\n")
    headings = paper_detect_headings(text)
    pieces, cursor = [], 0
    for heading in headings:
        body = "\n".join(lines[cursor:heading["line"] - 1]).strip()
        if body:
            pieces.append(clean_text(body))
        pieces.append(heading["title"])
        cursor = heading["line"]
    body = "\n".join(lines[cursor:]).strip()
    if body:
        pieces.append(clean_text(body))
    return "\n\n".join(pieces)


def paper_outline_text(headings):
    return "\n".join(f"{heading['level']} | {heading['title']}" for heading in headings)


def paper_parse_outline(text):
    headings = []
    for number, line in enumerate(text.splitlines(), 1):
        if not line.strip():
            continue
        level, separator, title = line.partition("|")
        if not separator or not level.strip().isdigit() or not 1 <= int(level.strip()) <= 6 or not title.strip():
            raise ValueError(f"Outline line {number}: use level | exact heading title (levels 1–6).")
        headings.append({"level": int(level.strip()), "title": title.strip()})
    return headings


def paper_partition(cleaned_text, headings):
    """Split own body text from child subsections; no span is counted twice."""
    text = normalize_newlines(cleaned_text)
    if not text.strip():
        raise ValueError("Paste or load paper text first.")
    lines = text.split("\n")
    located, cursor = [], 0
    for heading in headings:
        if type(heading.get("level")) is not int or not 1 <= heading["level"] <= 6 or not isinstance(heading.get("title"), str):
            raise ValueError("Invalid heading level or title.")
        title = heading["title"].strip()
        if not title or "\n" in title or "\r" in title:
            raise ValueError("A heading must be a nonempty single line.")
        found = next((i for i in range(cursor, len(lines)) if lines[i].strip() == title), None)
        if found is None:
            raise ValueError(f"Heading not found in order on its own line: {title!r}. Copy its exact title from Cleaned paper.")
        located.append({"title": title, "level": heading["level"], "line": found + 1})
        cursor = found + 1
    units = []
    first_heading = located[0]["line"] - 1 if located else len(lines)
    preamble = "\n".join(lines[:first_heading]).strip()
    if preamble:
        units.append({"title": "", "level": 0, "heading_line": None, "body": preamble})
    for index, heading in enumerate(located):
        end = located[index + 1]["line"] - 1 if index + 1 < len(located) else len(lines)
        units.append({"title": heading["title"], "level": heading["level"], "heading_line": heading["line"],
                      "body": "\n".join(lines[heading["line"]:end]).strip()})
    stack = []
    for index, unit in enumerate(units, 1):
        unit["id"] = f"part_{index:04d}"
        while stack and stack[-1]["level"] >= unit["level"]:
            stack.pop()
        unit["parent_id"] = stack[-1]["id"] if stack and unit["level"] else None
        unit["category"] = PAPER_UNASSIGNED
        if unit["level"]:
            stack.append(unit)
    return units


def paper_make_document(cleaned_text, outline_text, *, previous=None, include_heading_words=True,
                        paper_name="paper", source_text="", keep_hyphens=(), paper_metadata=None, project_metadata=None):
    text = normalize_newlines(cleaned_text)
    units = paper_partition(text, paper_parse_outline(outline_text))
    # Match repeated headings in document order when rebuilding after an edit.
    labels = defaultdict(deque)
    if previous:
        for unit in previous["units"]:
            labels[unit["title"]].append(unit["category"])
    for unit in units:
        if labels[unit["title"]]:
            unit["category"] = labels[unit["title"]].popleft()
    return {"schema_version": 1, "workflow": "full_paper_categorization", "paper_name": paper_name,
            "cleaned_text": text, "source_text": source_text, "keep_hyphens": sorted(set(keep_hyphens)),
            "include_heading_words": bool(include_heading_words), "units": units,
            "paper_metadata": dict(paper_metadata if paper_metadata is not None else (previous or {}).get("paper_metadata", {})),
            "project_metadata": dict(project_metadata if project_metadata is not None else (previous or {}).get("project_metadata", {}))}


def paper_unit_counts(unit, include_heading_words=True):
    body = count_text(unit["body"])
    return {"words": body["words"] + (len(WORD.findall(unit["title"])) if include_heading_words else 0),
            "paragraphs": body["paragraphs"]}


def paper_summary(document):
    rows = {category: {"category": category, "section_titles": [], "subsection_titles": [], "words": 0, "paragraphs": 0}
            for category in PAPER_LABELS}
    by_id = {unit["id"]: unit for unit in document["units"]}
    seen = {category: set() for category in PAPER_LABELS}
    for unit in document["units"]:
        row = rows[unit["category"]]
        counts = paper_unit_counts(unit, document["include_heading_words"])
        row["words"] += counts["words"]
        row["paragraphs"] += counts["paragraphs"]
        root = unit
        while root["parent_id"]:
            root = by_id[root["parent_id"]]
        if root["level"] <= 1 and root["id"] not in seen[unit["category"]]:
            row["section_titles"].append(root["title"] or "[Text before first heading / unheaded text]")
            seen[unit["category"]].add(root["id"])
        if unit["level"] > 1:
            row["subsection_titles"].append(unit["title"])
    return rows


def paper_validate_document(value):
    """Validate JSON before changing the editor; recompute structure and all counts."""
    if (not isinstance(value, dict) or type(value.get("schema_version")) is not int
            or value["schema_version"] != 1 or value.get("workflow") != "full_paper_categorization"):
        raise ValueError("This is not a supported paper-categorization JSON file (schema version 1).")
    if (not isinstance(value.get("cleaned_text"), str) or not isinstance(value.get("source_text", ""), str)
            or not isinstance(value.get("paper_name"), str) or type(value.get("include_heading_words")) is not bool
            or not isinstance(value.get("units"), list)):
        raise ValueError("The saved file has invalid paper text, settings or sections.")
    saved = value["units"]
    if not saved or any(not isinstance(unit, dict) for unit in saved):
        raise ValueError("The saved file has no valid sections.")
    outline = [{"title": unit.get("title"), "level": unit.get("level")} for unit in saved if unit.get("heading_line") is not None]
    units = paper_partition(value["cleaned_text"], outline)
    if len(saved) != len(units):
        raise ValueError("Saved section boundaries do not match the cleaned text.")
    for actual, expected in zip(saved, units):
        for field in ("id", "title", "level", "heading_line", "parent_id", "body"):
            if actual.get(field) != expected[field]:
                raise ValueError(f"Saved section structure does not match the cleaned text ({field}).")
        if actual.get("category") not in PAPER_LABELS:
            raise ValueError("The saved file contains an unknown category.")
        expected["category"] = actual["category"]
    keep = value.get("keep_hyphens", [])
    count = len(find_line_end_hyphens(value.get("source_text", "")))
    if not isinstance(keep, list) or any(type(index) is not int or not 0 <= index < count for index in keep):
        raise ValueError("The saved hyphen selection is invalid.")
    metadata = {}
    for field in ("paper_metadata", "project_metadata"):
        item = value.get(field, {})
        if not isinstance(item, dict) or any(not isinstance(key, str) or not isinstance(val, (str, int, bool, type(None))) for key, val in item.items()):
            raise ValueError("Invalid saved paper/project metadata.")
        metadata[field] = dict(item)
    return {"schema_version": 1, "workflow": "full_paper_categorization", "paper_name": value["paper_name"],
            "cleaned_text": normalize_newlines(value["cleaned_text"]), "source_text": value.get("source_text", ""),
            "keep_hyphens": sorted(set(keep)), "include_heading_words": value["include_heading_words"], "units": units, **metadata}


def paper_export_document(document):
    document = paper_validate_document(document)
    result = dict(document, saved_at_utc=datetime.now(timezone.utc).isoformat())
    result["counting"] = {"words": "Body words plus own heading words when enabled; existing WORD rule.",
                          "paragraphs": "Existing punctuation/footnote rule on own body text; all headings excluded.",
                          "section_titles": "Top-level titles provide context for assigned subsections, even when their parent has another label."}
    result["units"] = [dict(unit, counts=paper_unit_counts(unit, document["include_heading_words"])) for unit in document["units"]]
    result["summary"] = list(paper_summary(document).values())
    return result


def paper_save_files(document, directory, name, overwrite=False):
    """Save matching UTF-8 text/JSON; keep previous files if publication fails."""
    document = paper_export_document(document)
    name = name.strip()
    if not name or name in (".", "..") or any(char in name for char in '/\\\n\r\t<>:"|?*'):
        raise ValueError("Use a file prefix without folders or line breaks.")
    directory = Path(directory).expanduser().resolve()
    targets = (directory / (name + "_cleaned.txt"), directory / (name + "_categories.json"))
    for target in targets:
        if target.is_symlink() or (target.exists() and not target.is_file()):
            raise ValueError(f"Not a regular output file: {target}")
        if target.exists() and not overwrite:
            raise FileExistsError("Files already exist. Choose another prefix or enable Overwrite existing files.")
    directory.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix=".paper-save-", dir=directory) as temporary:
        stage = Path(temporary)
        contents = (document["cleaned_text"], json.dumps(document, ensure_ascii=False, indent=2) + "\n")
        for index, content in enumerate(contents):
            (stage / str(index)).write_text(content, encoding="utf-8")
        backups, published = {}, []
        try:
            for index, target in enumerate(targets):
                if target.exists():
                    backups[target] = stage / ("previous-" + str(index))
                    target.replace(backups[target])
            for index, target in enumerate(targets):
                (stage / str(index)).replace(target)
                published.append(target)
        except BaseException:
            for target in published:
                target.unlink()
            for target, previous in backups.items():
                previous.replace(target)
            raise
    return targets


PAPER_PROJECT_SAVE_FOLDER = "paper_categorizations"


def paper_read_project(directory):
    """Read an existing notebook-created project; never create folders while scanning."""
    root = Path(directory).expanduser().resolve()
    config = json.loads((root / "project.json").read_text(encoding="utf-8-sig"))
    if (not isinstance(config, dict) or config.get("schema_version") != 1
            or not isinstance(config.get("journal"), str) or not isinstance(config.get("sample_export"), str)):
        raise ValueError("This is not a supported journal-analysis project.json.")
    sample = (root / config["sample_export"]).resolve()
    if not sample.is_relative_to(root) or not sample.is_file():
        raise ValueError("The project's sampled-paper CSV is missing or outside its folder.")
    with sample.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        if not {"doi", "title"}.issubset(reader.fieldnames or []):
            raise ValueError("The sample CSV needs DOI and title columns.")
        papers = []
        seen = set()
        for number, row in enumerate(reader, 1):
            doi = (row.get("doi") or "").strip().lower()
            title = (row.get("title") or "").strip()
            if not doi or not title or doi in seen:
                raise ValueError("The sample CSV contains missing or duplicate paper identities.")
            seen.add(doi)
            paper = {key: (row.get(key) or "").strip() for key in ("title", "authors", "bibtex_key", "url")}
            paper.update(doi=doi, title=title, sample_number=number)
            papers.append(paper)
    save_directory = root / PAPER_PROJECT_SAVE_FOLDER
    if save_directory.exists() and (save_directory.is_symlink() or not save_directory.is_dir()):
        raise ValueError("The project's categorization folder must be a regular folder inside the project.")
    return {"root": root, "config": config, "papers": papers, "save_directory": save_directory}


def paper_find_project(directory):
    start = Path(directory).expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "project.json").is_file():
            return paper_read_project(candidate)
    return None


def paper_project_metadata(context):
    return {"journal": context["config"]["journal"], "sample_export": context["config"]["sample_export"]}


def paper_project_prefix(paper):
    doi = paper["doi"].strip().lower()
    readable = re.sub(r"[^a-z0-9]+", "_", doi).strip("_")[:90]
    suffix = hashlib.sha256(doi.encode("utf-8")).hexdigest()[:10]
    return f"paper_{readable}_{suffix}"


def paper_saved_in_project(context):
    """List supported categorization JSON anywhere inside the project, excluding metadata exports."""
    root = context["root"]
    saved = []
    for directory, folders, files in os.walk(root, followlinks=False):
        folders[:] = [name for name in folders if name not in {".git", ".venv", "venv", "__pycache__", ".ipynb_checkpoints"}]
        for name in files:
            if not name.lower().endswith(".json"):
                continue
            path = Path(directory) / name
            if path.is_symlink() or not path.resolve().is_relative_to(root):
                continue
            try:
                value = json.loads(path.read_text(encoding="utf-8-sig"))
                if not isinstance(value, dict) or value.get("workflow") != "full_paper_categorization":
                    continue
                saved.append({"path": path, "relative_path": path.relative_to(root).as_posix(),
                              "paper_name": str(value.get("paper_name", path.stem)),
                              "doi": str(value.get("paper_metadata", {}).get("doi", "")).lower(),
                              "title": str(value.get("paper_metadata", {}).get("title", "")),
                              "modified": path.stat().st_mtime})
            except (OSError, ValueError, TypeError, AttributeError):
                continue
    return sorted(saved, key=lambda item: (-item["modified"], item["relative_path"]))


def paper_save_in_project(document, context, name=None, overwrite=False):
    # Re-read the manifest and folder constraint at save time, not a disabled UI field.
    current = paper_read_project(context["root"])
    document = dict(document, project_metadata=paper_project_metadata(current))
    return paper_save_files(document, current["save_directory"], name or document["paper_name"], overwrite)


In [ ]:
# Small regression checks for the full-paper model.
_paper_example = "1. Introduction\nFirst paragraph?\n2. Methods\nAn overview.\n2.1 Data\nThe data are useful.\n2.2 Model\nWe estimate effects!\n3. Conclusion\nThe paper ends."
_paper_clean = paper_clean_text(_paper_example)
_paper_doc = paper_make_document(_paper_clean, paper_outline_text(paper_detect_headings(_paper_clean)), source_text=_paper_example)
assert len(_paper_doc["units"]) == 5
assert sum(paper_unit_counts(unit)["words"] for unit in _paper_doc["units"]) == len(WORD.findall(_paper_clean))
assert sum(paper_unit_counts(unit)["paragraphs"] for unit in _paper_doc["units"]) == 5
_paper_doc["units"][1]["category"] = "Methods"
_paper_doc["units"][2]["category"] = "Data"
assert paper_summary(_paper_doc)["Methods"]["paragraphs"] == 1  # No child double count.
assert paper_summary(_paper_doc)["Data"]["section_titles"] == ["2. Methods"]
assert paper_summary(_paper_doc)["Data"]["subsection_titles"] == ["2.1 Data"]
assert paper_validate_document(json.loads(json.dumps(paper_export_document(_paper_doc)))) == _paper_doc
assert paper_clean_text("Result (𝑝<.1\n).\nNext!") == "Result (𝑝<.1).\n\nNext!"
assert "Introduction\n\n" in paper_clean_text("Introduction\nWe ask why?\nMethods\nWe test it.")
assert "well-known" in paper_clean_text("1. Introduction\nA well-\nknown result.", keep_hyphens={0})
assert not paper_detect_headings("The sample is useful.12")
assert not paper_detect_headings("The result has p < .1")
print("Full-paper model checks passed. Run the next cell to open both workflows.")


In [ ]:
# Independent full-paper editor. The original workflow keeps its own widgets/state.
paper_style = {"description_width": "140px"}
paper_wide = widgets.Layout(width="100%")
paper_original = widgets.Textarea(description="Original paper:", placeholder="Paste the full paper, excluding table text, figure notes and formulas.",
    style=paper_style, layout=widgets.Layout(width="100%", height="240px"))
paper_cleaned = widgets.Textarea(description="Cleaned paper:", placeholder="Cleaned text remains editable here.",
    style=paper_style, layout=widgets.Layout(width="100%", height="300px"))
paper_keep = widgets.SelectMultiple(description="Keep hyphens:", options=[], rows=5, style=paper_style, layout=paper_wide)
paper_find_button = widgets.Button(description="Find line-end hyphens", layout=widgets.Layout(width="200px"))
paper_clean_button = widgets.Button(description="Clean full paper", button_style="primary", layout=widgets.Layout(width="170px"))
paper_clear_button = widgets.Button(description="Clear original paper", button_style="danger", layout=widgets.Layout(width="185px", margin="0 0 0 32px"))
paper_status = widgets.HTML()
paper_outline = widgets.Textarea(description="Heading outline:", placeholder="1 | 1. Introduction\n2 | 1.1 Motivation\n1 | 2. Methods",
    style=paper_style, layout=widgets.Layout(width="100%", height="180px"))
paper_detect_button = widgets.Button(description="Redetect headings", layout=widgets.Layout(width="180px"))
paper_rebuild_button = widgets.Button(description="Apply reviewed outline", button_style="info", layout=widgets.Layout(width="200px"))
paper_heading_options = widgets.Accordion(children=[widgets.VBox([
    widgets.HTML("<p>Review the detected headings in document order. Each line is <code>level | exact heading title</code>: "
                 "1 = section; 2 or higher = subsection. Add missed headings, remove false matches, or adjust levels. "
                 "Each title must occur on its own line in Cleaned paper. Repeated titles are matched in order.</p>"),
    paper_outline, widgets.HBox([paper_detect_button, paper_rebuild_button]),
])], selected_index=None)
paper_heading_options.set_title(0, "Review / correct heading boundaries")
paper_include_headings = widgets.Checkbox(value=True, description="Include heading words in word counts", indent=False, layout=paper_wide)
paper_suggest_button = widgets.Button(description="Fill labels from titles / parents", layout=widgets.Layout(width="260px"), disabled=True)
paper_section_rows = widgets.VBox()
paper_preview = widgets.Textarea(description="Section text:", value="", disabled=True, style=paper_style,
    layout=widgets.Layout(width="100%", height="170px"))
paper_preview_title = widgets.HTML(layout=widgets.Layout(min_width="0", flex="1 1 auto", max_height="80px", overflow="hidden auto"))
paper_hide_text_button = widgets.Button(description="Hide text", layout=widgets.Layout(width="110px", flex="0 0 110px"))
paper_preview_panel = widgets.VBox([
    widgets.HBox([paper_preview_title, paper_hide_text_button], layout=widgets.Layout(justify_content="space-between", width="100%")),
    paper_preview,
], layout=widgets.Layout(width="100%", display="none"))
paper_preview_panel.add_class("paper-preview-panel")
paper_summary_css = widgets.HTML("""<style>
.paper-summary-cell, .paper-summary-content, .paper-summary-cell > .widget-button {
  margin: 0 !important; box-sizing: border-box !important;
}
.paper-summary-cell { border-bottom: 1px solid #d9dfe6; }
.paper-summary-content .widget-html-content {
  width: 100%; min-width: 0; white-space: normal; overflow-wrap: anywhere;
}
.paper-section-row { border: 2px solid transparent; box-sizing: border-box; }
.paper-section-row.paper-unassigned { border-color: #b77900; background: #fff4cf; }
.paper-completion-warning .widget-html-content { width: 100%; min-width: 0; }
.paper-preview-panel {
  position: fixed; right: 24px; bottom: 24px; z-index: 1000;
  width: min(720px, calc(100vw - 48px)) !important; max-height: 60vh;
  box-sizing: border-box; padding: 12px; background: white;
  border: 1px solid #c5ccd4; box-shadow: 0 4px 20px #0003;
}
.paper-preview-panel .widget-html-content { overflow-wrap: anywhere; min-width: 0; }
</style>""", layout=widgets.Layout(display="none"))
paper_summary_box = widgets.VBox(
    [widgets.HTML("<p>Clean a paper to create a section outline and category summary.</p>")],
    layout=widgets.Layout(width="100%", min_width="0", overflow="auto hidden"))
paper_completion_warning = widgets.HTML(
    "<p>Clean a paper to check category assignments.</p>",
    layout=widgets.Layout(width="100%", min_width="0", height="76px", min_height="76px", max_height="76px", overflow="hidden auto"))
paper_completion_warning.add_class("paper-completion-warning")
paper_totals = widgets.HTML()
paper_copy_table_button = widgets.Button(description="Copy whole summary (Excel)", layout=widgets.Layout(width="240px"), disabled=True)
paper_clipboard_status = widgets.HTML()
paper_copy_fallback = widgets.Textarea(description="Copy manually:", style=paper_style, disabled=False,
    layout=widgets.Layout(width="100%", height="90px", display="none"))
paper_name = widgets.Text(description="File prefix / paper:", value="paper", style=paper_style, layout=paper_wide)
paper_folder = widgets.Text(description="Save folder:", value=str(Path.cwd().parent / "paper_categorizations"), style=paper_style, layout=paper_wide)
paper_overwrite = widgets.Checkbox(value=False, description="Overwrite existing files", indent=False)
paper_save_button = widgets.Button(description="Save cleaned text + JSON", button_style="success", disabled=True,
    layout=widgets.Layout(width="230px"))
paper_save_status = widgets.HTML()
paper_upload = widgets.FileUpload(description="Load saved JSON", accept=".json", multiple=False, layout=widgets.Layout(width="190px"))
paper_json_path = widgets.Text(description="JSON path:", placeholder="Or enter a saved JSON path on the notebook computer", style=paper_style, layout=paper_wide)
paper_load_path_button = widgets.Button(description="Load JSON from path", layout=widgets.Layout(width="190px"))
paper_project_folder = widgets.Text(description="Project folder:", value=str(Path.cwd()), style=paper_style, layout=paper_wide)
paper_scan_project_button = widgets.Button(description="Scan project folder", layout=widgets.Layout(width="190px"))
paper_project_status = widgets.HTML()
paper_sample_choice = widgets.Dropdown(description="Sampled paper:", options=[("No project selected", None)], value=None,
    style=paper_style, layout=widgets.Layout(width="100%", min_width="0"))
paper_use_sample_button = widgets.Button(description="Start / reopen paper", disabled=True, layout=widgets.Layout(width="190px"))
paper_associate_button = widgets.Button(description="Associate current text", disabled=True, layout=widgets.Layout(width="210px"))
paper_saved_choice = widgets.Dropdown(description="Saved JSON:", options=[("No saved categorizations", None)], value=None,
    style=paper_style, layout=widgets.Layout(width="100%", min_width="0"))
paper_load_saved_button = widgets.Button(description="Load selected JSON", disabled=True, layout=widgets.Layout(width="190px"))
paper_replace_current = widgets.Checkbox(value=False, description="Allow replacing the current unsaved paper", indent=False, layout=paper_wide)
paper_active_info = widgets.HTML()
paper_state = {"document": None, "reviewed_source": None, "updating": False, "dropdowns": {}, "summary_grid": None, "summary_cells": {}, "project": None, "saved_files": [], "active_paper": None,
               "unsaved": False, "preview_unit": None, "preview_buttons": {}, "section_widgets": {}}


def paper_is_current():
    document = paper_state["document"]
    if document is None:
        return False
    headings = [{"title": unit["title"], "level": unit["level"]} for unit in document["units"] if unit["heading_line"] is not None]
    return paper_cleaned.value == document["cleaned_text"] and paper_outline.value == paper_outline_text(headings)


def paper_copy_value(value, label):
    value = str(value)
    utility = {"darwin": "pbcopy", "win32": "clip.exe"}.get(sys.platform)
    command = shutil.which(utility) if utility else None
    try:
        if not command:
            raise OSError("No native clipboard utility")
        encoding = "utf-16-le" if sys.platform == "win32" else "utf-8"
        options = {"creationflags": subprocess.CREATE_NO_WINDOW} if sys.platform == "win32" else {}
        subprocess.run([command], input=value.encode(encoding), capture_output=True, check=True, timeout=5, **options)
    except (OSError, subprocess.SubprocessError):
        paper_copy_fallback.value = value
        paper_copy_fallback.layout.display = ""
        paper_clipboard_status.value = "<p>Clipboard unavailable. Select the value in Copy manually and copy it.</p>"
    else:
        paper_copy_fallback.layout.display = "none"
        paper_clipboard_status.value = f"<p>Copied {escape(label)}.</p>"


def paper_titles_cell(titles, separator="\n"):
    # Individual title buttons copy one title per line.
    return separator.join(" ".join(title.split()) for title in titles)


def paper_copy_summary(button=None):
    if not paper_is_current():
        return
    rows = paper_summary(paper_state["document"])
    lines = ["Category\tSection titles\tSubsection titles\tWords\tParagraphs"]
    for category in PAPER_CATEGORIES:
        row = rows[category]
        lines.append("\t".join([category, paper_titles_cell(row["section_titles"], "; "), paper_titles_cell(row["subsection_titles"], "; "),
                                str(row["words"]), str(row["paragraphs"])]))
    paper_copy_value("\n".join(lines), "category summary")


def paper_copy_summary_cell(category, key):
    """Resolve the current value when clicked; never capture an outdated count."""
    if not paper_is_current():
        return
    row = paper_summary(paper_state["document"])[category]
    value = paper_titles_cell(row[key]) if isinstance(row[key], list) else str(row[key])
    label = {"section_titles": "section titles", "subsection_titles": "subsection titles",
             "words": "word count", "paragraphs": "paragraph count"}[key]
    paper_copy_value(value, category + " " + label)


def paper_ensure_summary_grid():
    """Create the table once. Fixed cell sizes keep all copy controls anchored."""
    if paper_state["summary_grid"] is not None:
        return paper_state["summary_grid"]
    columns = ("Category", "Section titles", "Subsection titles", "Words", "Paragraphs")
    cells = []
    for label in columns:
        header = widgets.HTML(f"<strong>{escape(label)}</strong>",
            layout=widgets.Layout(width="100%", min_width="0", height="36px", overflow="hidden", padding="8px"))
        header.add_class("paper-summary-cell")
        cells.append(header)
    fields = {}
    for category in PAPER_CATEGORIES:
        category_cell = widgets.HTML(f"<strong>{escape(category)}</strong>", layout=widgets.Layout(
            width="100%", min_width="0", height="156px", padding="8px", overflow="hidden"))
        category_cell.add_class("paper-summary-cell")
        cells.append(category_cell)
        for key in ("section_titles", "subsection_titles", "words", "paragraphs"):
            value = widgets.HTML("—", layout=widgets.Layout(
                width="100%", min_width="0", max_width="100%", height="102px", min_height="102px",
                max_height="102px", overflow="hidden auto"))
            value.add_class("paper-summary-content")
            button = widgets.Button(description="Copy titles" if key.endswith("titles") else "Copy count",
                tooltip=category + " — " + key.replace("_", " "),
                layout=widgets.Layout(width="100%", min_width="0", max_width="110px",
                                      height="32px", min_height="32px", max_height="32px", flex="0 0 32px"))
            button.on_click(lambda _, cat=category, field=key: paper_copy_summary_cell(cat, field))
            box = widgets.VBox([value, button], layout=widgets.Layout(
                width="100%", min_width="0", height="156px", min_height="156px", max_height="156px",
                padding="8px", overflow="hidden", justify_content="space-between"))
            box.add_class("paper-summary-cell")
            cells.append(box)
            fields[(category, key)] = (value, button)
    grid = widgets.GridBox(children=cells, layout=widgets.Layout(
        grid_template_columns="150px 250px 300px 120px 130px",
        grid_template_rows=f"36px repeat({len(PAPER_CATEGORIES)}, 156px)", grid_gap="0px 8px",
        width="982px", min_width="982px", max_width="982px", overflow="hidden"))
    grid.add_class("paper-summary-grid")
    paper_state["summary_cells"] = fields
    paper_state["summary_grid"] = grid
    return grid


def paper_refresh_summary():
    if not paper_is_current():
        return
    document = paper_state["document"]
    paper_copy_fallback.layout.display = "none"
    paper_clipboard_status.value = ""
    rows = paper_summary(document)
    grid = paper_ensure_summary_grid()
    for category in PAPER_CATEGORIES:
        row = rows[category]
        for key in ("section_titles", "subsection_titles", "words", "paragraphs"):
            value, button = paper_state["summary_cells"][(category, key)]
            items = row[key]
            value.value = ("<br>".join(escape(title) for title in items) or "—") if isinstance(items, list) else escape(str(items))
            button.disabled = isinstance(items, list) and not items
    # Reuse the same grid/model objects even after edits or loading another paper.
    if paper_summary_box.children != (grid,):
        paper_summary_box.children = (grid,)
    assigned = sum(rows[category]["words"] for category in PAPER_CATEGORIES)
    unassigned = rows[PAPER_UNASSIGNED]
    excluded = rows[PAPER_EXCLUDED]
    total_words = sum(row["words"] for row in rows.values())
    total_pars = sum(row["paragraphs"] for row in rows.values())
    remaining = sum(unit["category"] == PAPER_UNASSIGNED for unit in document["units"])
    for unit in document["units"]:
        row_widget = paper_state["section_widgets"].get(unit["id"])
        if row_widget is not None:
            if unit["category"] == PAPER_UNASSIGNED:
                row_widget.add_class("paper-unassigned")
            else:
                row_widget.remove_class("paper-unassigned")
    if remaining:
        paper_completion_warning.value = (
            f'<p role="alert" style="margin:4px 0;padding:8px;background:#fff4cf;color:#654700;">'
            f'<strong>Warning: summary incomplete.</strong> {remaining} section(s) / subsection(s) are still Unassigned. '
            'Review the highlighted rows above. Assign a category, Not categorizable, or Exclude from summary.</p>')
    else:
        paper_completion_warning.value = (
            '<p role="status" style="margin:4px 0;padding:8px;background:#e8f4e9;color:#235c2b;">'
            '<strong>All sections have been categorized.</strong> No unassigned sections remain.</p>')
    paper_totals.value = (
        f"<p><strong>Entire paper:</strong> {total_words} words; {total_pars} paragraphs. "
        f"<strong>Assigned to the {len(PAPER_CATEGORIES)} categories:</strong> {assigned} words. "
        f"<strong>Unassigned:</strong> {remaining} part(s), {unassigned['words']} words / {unassigned['paragraphs']} paragraphs. "
        f"<strong>Excluded:</strong> {excluded['words']} words / {excluded['paragraphs']} paragraphs.</p>"
    )
    paper_save_button.disabled = False
    paper_copy_table_button.disabled = False
    paper_suggest_button.disabled = False


def paper_hide_preview(button=None):
    paper_preview.value = ""
    paper_preview_title.value = ""
    paper_preview_panel.layout.display = "none"
    paper_state["preview_unit"] = None
    for control in paper_state["preview_buttons"].values():
        control.description = "Show text"


def paper_show_unit(unit_id):
    if paper_state["preview_unit"] == unit_id:
        paper_hide_preview()
        return
    paper_hide_preview()
    unit = next(unit for unit in paper_state["document"]["units"] if unit["id"] == unit_id)
    paper_preview.value = unit["body"]
    paper_preview_title.value = f"<strong>{escape(unit['title'] or 'Unheaded text')}</strong>"
    paper_preview_panel.layout.display = "flex"
    paper_state["preview_unit"] = unit_id
    paper_state["preview_buttons"][unit_id].description = "Hide text"


def paper_category_changed(change, unit_id):
    if paper_state["updating"] or not paper_is_current():
        return
    unit = next(unit for unit in paper_state["document"]["units"] if unit["id"] == unit_id)
    unit["category"] = change["new"]
    paper_state["unsaved"] = True
    paper_save_status.value = "<p>Categorization changed; save to update the JSON file.</p>"
    paper_refresh_summary()


def paper_assign_descendants(unit_id):
    document = paper_state["document"]
    if not paper_is_current():
        return
    units = document["units"]
    index = next(index for index, unit in enumerate(units) if unit["id"] == unit_id)
    parent = units[index]
    paper_state["updating"] = True
    try:
        for child in units[index + 1:]:
            if child["level"] <= parent["level"]:
                break
            child["category"] = parent["category"]
            paper_state["dropdowns"][child["id"]].value = parent["category"]
    finally:
        paper_state["updating"] = False
    paper_state["unsaved"] = True
    paper_save_status.value = "<p>Assigned this section's label to all its subsections. Save to keep these changes.</p>"
    paper_refresh_summary()


def paper_render_sections():
    document = paper_state["document"]
    paper_hide_preview()
    rows, dropdowns, preview_buttons, section_widgets = [], {}, {}, {}
    for index, unit in enumerate(document["units"]):
        title = unit["title"] or "[Text before first heading / unheaded text]"
        kind = "Section" if unit["level"] == 1 else (f"Subsection (level {unit['level']})" if unit["level"] else "Unheaded text")
        counts = paper_unit_counts(unit, document["include_heading_words"])
        title_widget = widgets.HTML(f"<strong>{escape(title)}</strong><br><small>{kind}: {counts['words']} words; {counts['paragraphs']} paragraphs</small>",
                                    layout=widgets.Layout(width="42%"))
        dropdown = widgets.Dropdown(options=PAPER_LABELS, value=unit["category"], layout=widgets.Layout(width="220px"))
        dropdown.observe(lambda change, uid=unit["id"]: paper_category_changed(change, uid), names="value")
        dropdowns[unit["id"]] = dropdown
        preview_button = widgets.Button(description="Show text", layout=widgets.Layout(width="100px"))
        preview_button.on_click(lambda _, uid=unit["id"]: paper_show_unit(uid))
        preview_buttons[unit["id"]] = preview_button
        children = [title_widget, dropdown, preview_button]
        has_children = unit["level"] > 0 and index + 1 < len(document["units"]) and document["units"][index + 1]["level"] > unit["level"]
        if has_children:
            apply_button = widgets.Button(description="Apply to subsections", layout=widgets.Layout(width="180px"))
            apply_button.on_click(lambda _, uid=unit["id"]: paper_assign_descendants(uid))
            children.append(apply_button)
        rows.append(widgets.HBox(children, layout=widgets.Layout(
            margin=f"3px 0 3px {max(0, unit['level'] - 1) * 16}px", align_items="center", flex_flow="row wrap")))
        rows[-1].add_class("paper-section-row")
        section_widgets[unit["id"]] = rows[-1]
    paper_state["section_widgets"] = section_widgets
    paper_state["preview_buttons"] = preview_buttons
    paper_state["dropdowns"] = dropdowns
    paper_section_rows.children = tuple(rows)
    paper_refresh_summary()


def paper_structure_changed(change=None):
    if paper_state["updating"] or paper_state["document"] is None:
        return
    if paper_is_current():
        paper_render_sections()
        return
    paper_section_rows.children = ()
    paper_summary_box.children = (widgets.HTML("<p>Cleaned text or heading boundaries changed. Click <strong>Apply reviewed outline</strong> to update the sections and counts.</p>"),)
    paper_completion_warning.value = "<p>Apply the reviewed outline to check category assignments.</p>"
    paper_totals.value = ""
    paper_hide_preview()
    paper_state["unsaved"] = True
    paper_save_button.disabled = True
    paper_copy_table_button.disabled = True
    paper_suggest_button.disabled = True
    paper_save_status.value = "<p>Apply the reviewed outline before saving. Labels will be retained for matching headings.</p>"


def paper_rebuild_sections(button=None, previous=None):
    try:
        previous = paper_state["document"] if previous is None else previous
        document = paper_make_document(paper_cleaned.value, paper_outline.value, previous=previous,
            include_heading_words=paper_include_headings.value, paper_name=paper_name.value,
            source_text=previous["source_text"] if previous else paper_original.value,
            keep_hyphens=previous["keep_hyphens"] if previous else paper_keep.value,
            paper_metadata=paper_state["active_paper"] or (previous or {}).get("paper_metadata", {}),
            project_metadata=paper_project_metadata(paper_state["project"]) if paper_state["project"] else (previous or {}).get("project_metadata", {}))
    except (ValueError, TypeError) as error:
        paper_status.value = f"<p>{escape(str(error))}</p>"
        return False
    paper_state["updating"] = True
    try:
        paper_state["document"] = document
        paper_cleaned.value = document["cleaned_text"]
        paper_outline.value = paper_outline_text([unit for unit in document["units"] if unit["heading_line"] is not None])
    finally:
        paper_state["updating"] = False
    paper_render_sections()
    paper_status.value = f"<p>{len(document['units'])} part(s) ready. Review the heading outline, then assign each part or exclude it from the summary.</p>"
    paper_state["unsaved"] = True
    paper_save_status.value = "<p>Unsaved categorization.</p>"
    return True


def paper_find_splits(button=None):
    splits = find_line_end_hyphens(paper_original.value)
    paper_keep.options = [(f"Line {item['line']}: {item['left']}- | {item['right']} → {item['joined']} "
                           f"(select to keep {item['hyphenated']})", item["index"]) for item in splits]
    paper_keep.value = ()
    paper_state["reviewed_source"] = paper_original.value
    paper_status.value = f"<p>Found {len(splits)} split(s). Select genuine hyphens, then click Clean full paper.</p>"


def paper_apply_cleanup(button=None):
    if not paper_original.value.strip():
        paper_status.value = "<p>Paste the full paper first.</p>"
        return
    if paper_state["reviewed_source"] != paper_original.value:
        paper_find_splits()
        if paper_keep.options:
            return
    text = paper_clean_text(paper_original.value, paper_keep.value)
    paper_state["updating"] = True
    try:
        # A cleanup starts a fresh paper; rebuilding a reviewed outline preserves labels instead.
        paper_state["document"] = None
        paper_cleaned.value = text
        paper_outline.value = paper_outline_text(paper_detect_headings(text))
    finally:
        paper_state["updating"] = False
    paper_rebuild_sections()


def paper_clear_original(button=None):
    paper_original.value = ""
    paper_keep.options = []
    paper_keep.value = ()
    paper_state["reviewed_source"] = None
    paper_status.value = "<p>Original paper cleared. The current cleaned text and categorization are retained.</p>"


def paper_redetect(button=None):
    paper_outline.value = paper_outline_text(paper_detect_headings(paper_cleaned.value))
    paper_heading_options.selected_index = 0
    paper_status.value = "<p>Review the suggested boundaries, then click Apply reviewed outline.</p>"


def paper_heading_words_changed(change):
    if paper_state["updating"] or paper_state["document"] is None:
        return
    paper_state["document"]["include_heading_words"] = change["new"]
    if paper_is_current():
        paper_render_sections()
    paper_state["unsaved"] = True
    paper_save_status.value = "<p>Counting option changed; save to update the JSON file.</p>"


def paper_suggest_labels(button=None):
    if not paper_is_current():
        return
    synonyms = {"conclusions": "Conclusion", "concluding remarks": "Conclusion", "related literature": "Literature Review",
                "related work": "Literature Review", "theoretical framework": "Theory", "data description": "Data",
                "empirical methods": "Methods", "empirical strategy": "Methods", "empirical results": "Results"}
    exact = {category.casefold(): category for category in PAPER_CATEGORIES}
    exact.update(synonyms)
    document = paper_state["document"]
    by_id = {unit["id"]: unit for unit in document["units"]}
    for unit in document["units"]:
        if unit["category"] != PAPER_UNASSIGNED:
            continue
        numbered = PAPER_NUMBERED_TITLE.fullmatch(unit["title"])
        name = numbered.group(2) if numbered else unit["title"]
        suggestion = exact.get(" ".join(name.rstrip(":").casefold().split()))
        if suggestion is None and unit["parent_id"]:
            parent = by_id[unit["parent_id"]]
            suggestion = parent["category"] if parent["category"] in PAPER_CATEGORIES else None
        if suggestion:
            unit["category"] = suggestion
    paper_render_sections()
    paper_state["unsaved"] = True
    paper_save_status.value = "<p>Filled unassigned labels from exact/common titles and labelled parents. Review every label and save.</p>"


def paper_save(button=None):
    try:
        if not paper_is_current():
            raise ValueError("Apply the reviewed outline before saving.")
        if not paper_folder.value.strip():
            raise ValueError("Enter a save folder on the notebook computer.")
        document = dict(paper_state["document"], paper_name=paper_name.value)
        context = paper_state["project"]
        if context:
            txt, data = paper_save_in_project(document, context, paper_name.value, paper_overwrite.value)
            paper_folder.value = str(context["save_directory"])
            paper_state["document"]["project_metadata"] = paper_project_metadata(context)
        else:
            txt, data = paper_save_files(document, paper_folder.value, paper_name.value, paper_overwrite.value)
        paper_state["document"]["paper_name"] = paper_name.value
    except (OSError, ValueError, TypeError) as error:
        paper_save_status.value = f"<p>{escape(str(error))}</p>"
    else:
        paper_state["unsaved"] = False
        paper_refresh_project_lists(selected_json=str(data))
        paper_save_status.value = f"<p>Saved cleaned text: <code>{escape(str(txt))}</code><br>Saved reloadable JSON: <code>{escape(str(data))}</code></p>"


def paper_load_document(value):
    document = paper_validate_document(value)
    paper_state["updating"] = True
    try:
        paper_state["document"] = document
        paper_original.value = document["source_text"]
        paper_find_splits()
        paper_keep.value = tuple(document["keep_hyphens"])
        paper_cleaned.value = document["cleaned_text"]
        paper_outline.value = paper_outline_text([unit for unit in document["units"] if unit["heading_line"] is not None])
        paper_include_headings.value = document["include_heading_words"]
        paper_name.value = document["paper_name"]
    finally:
        paper_state["updating"] = False
    paper_state["unsaved"] = False
    paper_sync_active_from_document()
    if paper_state["project"]:
        paper_folder.value = str(paper_state["project"]["save_directory"])
    paper_render_sections()
    paper_status.value = "<p>Paper loaded. Change any category and save again when ready.</p>"
    paper_save_status.value = "<p>Loaded categorization; counts were recomputed from the cleaned text.</p>"


def paper_upload_changed(change):
    value = change["new"]
    if not value:
        return
    try:
        item = next(iter(value.values())) if isinstance(value, dict) else value[0]
        data = json.loads(bytes(item["content"]).decode("utf-8-sig"))
        if not paper_can_replace():
            return
        paper_load_document(data)
    except (OSError, ValueError, TypeError, KeyError) as error:
        paper_status.value = f"<p>Could not load paper: {escape(str(error))}</p>"


def paper_load_path(button=None):
    try:
        path = Path(paper_json_path.value.strip()).expanduser()
        data = json.loads(path.read_text(encoding="utf-8-sig"))
        if not paper_can_replace():
            return
        paper_load_document(data)
        if not paper_state["project"]:
            paper_folder.value = str(path.resolve().parent)
    except (OSError, ValueError, TypeError, KeyError) as error:
        paper_status.value = f"<p>Could not load paper: {escape(str(error))}</p>"


def paper_has_unsaved_work():
    document = paper_state["document"]
    return paper_state["unsaved"] or bool(paper_original.value.strip() and
        (document is None or paper_original.value != document["source_text"]))


def paper_can_replace():
    if paper_has_unsaved_work() and not paper_replace_current.value:
        paper_status.value = "<p>Save the current paper first, or enable Allow replacing the current unsaved paper before switching/reloading.</p>"
        return False
    return True


def paper_sync_active_from_document():
    context = paper_state["project"]
    metadata = (paper_state["document"] or {}).get("paper_metadata", {})
    doi = str(metadata.get("doi", "")).lower()
    active = next((item for item in context["papers"] if item["doi"] == doi), None) if context else None
    paper_state["active_paper"] = active
    if context:
        paper_sample_choice.value = active["doi"] if active else None
    paper_active_info.value = (f"<p><strong>Active paper:</strong> {escape(active['title'])}<br>DOI: {escape(active['doi'])}</p>"
                               if active else "")


def paper_refresh_project_lists(selected_json=None):
    context = paper_state["project"]
    if not context:
        return
    previous = paper_sample_choice.value
    papers = context["papers"]
    paper_sample_choice.options = [("Choose a sampled paper…", None)] + [(f"{item['sample_number']}. {item['title']}", item["doi"]) for item in papers]
    if previous in [item["doi"] for item in papers]:
        paper_sample_choice.value = previous
    saved = paper_saved_in_project(context)
    paper_state["saved_files"] = saved
    previous_file = selected_json or paper_saved_choice.value
    paper_saved_choice.options = [("Choose a saved categorization…", None)] + [
        (f"{item['title'] or item['paper_name']} — {item['relative_path']}", str(item["path"])) for item in saved]
    if previous_file in [str(item["path"]) for item in saved]:
        paper_saved_choice.value = previous_file
    paper_use_sample_button.disabled = not papers
    paper_associate_button.disabled = not papers
    paper_load_saved_button.disabled = not saved
    paper_project_status.value = (
        f"<p><strong>Project:</strong> {escape(context['config']['journal'])}. {len(papers)} sampled paper(s); "
        f"{len(saved)} saved categorization(s).<br>All saves go to <code>{escape(str(context['save_directory']))}</code>.</p>")


def paper_scan_project(button=None):
    try:
        context = paper_find_project(paper_project_folder.value.strip() or Path.cwd())
        # A notebook inside an actual project cannot accidentally unlock its save location.
        if context is None:
            context = paper_find_project(Path.cwd())
    except (OSError, ValueError, TypeError, KeyError) as error:
        paper_project_status.value = f"<p>Could not scan project: {escape(str(error))}</p>"
        return
    paper_state["project"] = context
    paper_folder.disabled = context is not None
    if context:
        paper_project_folder.value = str(context["root"])
        paper_folder.value = str(context["save_directory"])
        paper_refresh_project_lists()
        paper_sync_active_from_document()
    else:
        paper_sample_choice.options = [("No project selected", None)]
        paper_saved_choice.options = [("No project selected", None)]
        paper_state["saved_files"] = []
        paper_state["active_paper"] = None
        paper_use_sample_button.disabled = paper_associate_button.disabled = paper_load_saved_button.disabled = True
        paper_active_info.value = ""
        paper_project_status.value = "<p>Standalone mode. For a notebook-created project, enter its folder and click Scan project folder. Otherwise choose your save folder below.</p>"


def paper_chosen_sample():
    context = paper_state["project"]
    return next((item for item in context["papers"] if item["doi"] == paper_sample_choice.value), None) if context else None


def paper_associate_sample(button=None):
    selected = paper_chosen_sample()
    if selected is None:
        paper_status.value = "<p>Select a sampled paper first.</p>"
        return
    paper_state["active_paper"] = selected
    paper_name.value = paper_project_prefix(selected)
    if paper_state["document"]:
        paper_state["document"]["paper_metadata"] = dict(selected)
        paper_state["document"]["project_metadata"] = paper_project_metadata(paper_state["project"])
        paper_state["document"]["paper_name"] = paper_name.value
    paper_state["unsaved"] = bool(paper_original.value.strip() or paper_cleaned.value.strip())
    paper_active_info.value = f"<p><strong>Active paper:</strong> {escape(selected['title'])}<br>DOI: {escape(selected['doi'])}</p>"
    paper_save_status.value = "<p>Current text associated with the selected paper. Save to keep this identity and categorization.</p>"


def paper_start_sample(button=None):
    selected = paper_chosen_sample()
    if selected is None:
        paper_status.value = "<p>Select a sampled paper first.</p>"
        return
    active = paper_state["active_paper"]
    if active and active["doi"] == selected["doi"] and (paper_original.value.strip() or paper_cleaned.value.strip()):
        paper_status.value = "<p>This paper is already active. Use Saved JSON to load a particular saved version.</p>"
        return
    if not paper_can_replace():
        return
    saved = next((item for item in paper_state["saved_files"] if item["doi"] == selected["doi"]), None)
    if saved:
        try:
            paper_load_document(json.loads(saved["path"].read_text(encoding="utf-8-sig")))
        except (OSError, ValueError, TypeError, KeyError) as error:
            paper_status.value = f"<p>Could not reopen paper: {escape(str(error))}</p>"
        return
    paper_state["updating"] = True
    try:
        paper_state["document"] = None
        paper_original.value = paper_cleaned.value = paper_outline.value = ""
        paper_keep.options = []
        paper_state["reviewed_source"] = None
        paper_section_rows.children = ()
        paper_hide_preview()
        paper_save_button.disabled = paper_copy_table_button.disabled = paper_suggest_button.disabled = True
        paper_completion_warning.value = "<p>Apply the reviewed outline to check category assignments.</p>"
        paper_totals.value = ""
        # Keep the fixed-size summary grid anchored when changing papers.
        grid = paper_ensure_summary_grid()
        for value, control in paper_state["summary_cells"].values():
            value.value = "—"
            control.disabled = True
        paper_summary_box.children = (grid,)
    finally:
        paper_state["updating"] = False
    paper_associate_sample()
    paper_state["unsaved"] = False
    paper_status.value = "<p>Paper selected. Paste its full text above, then clean and categorize it.</p>"


def paper_load_saved(button=None):
    path = paper_saved_choice.value
    if not path:
        paper_status.value = "<p>Select a saved categorization first.</p>"
        return
    if not paper_can_replace():
        return
    try:
        paper_load_document(json.loads(Path(path).read_text(encoding="utf-8-sig")))
    except (OSError, ValueError, TypeError, KeyError) as error:
        paper_status.value = f"<p>Could not load paper: {escape(str(error))}</p>"


paper_hide_text_button.on_click(paper_hide_preview)
paper_scan_project_button.on_click(paper_scan_project)
paper_use_sample_button.on_click(paper_start_sample)
paper_associate_button.on_click(paper_associate_sample)
paper_load_saved_button.on_click(paper_load_saved)
paper_scan_project()
paper_find_button.on_click(paper_find_splits)
paper_clean_button.on_click(paper_apply_cleanup)
paper_clear_button.on_click(paper_clear_original)
paper_detect_button.on_click(paper_redetect)
paper_rebuild_button.on_click(paper_rebuild_sections)
paper_suggest_button.on_click(paper_suggest_labels)
paper_copy_table_button.on_click(paper_copy_summary)
paper_save_button.on_click(paper_save)
paper_load_path_button.on_click(paper_load_path)
paper_upload.observe(paper_upload_changed, names="value")
paper_cleaned.observe(paper_structure_changed, names="value")
paper_outline.observe(paper_structure_changed, names="value")
paper_include_headings.observe(paper_heading_words_changed, names="value")

paper_workflow = widgets.VBox([
    widgets.HTML("<h3>Categorize a full paper</h3><p>Paste prose with section headings, excluding table text, figure notes and formulas. "
                 "Clean, review the outline, assign labels, then save. Applying cleanup starts a fresh categorization; save before processing another paper.</p>"),
    paper_original,
    widgets.HBox([widgets.HBox([paper_find_button, paper_clean_button]), paper_clear_button],
                 layout=widgets.Layout(width="100%", justify_content="space-between")),
    paper_keep, paper_status, paper_cleaned, paper_heading_options,
    widgets.HTML("<h4>Label sections and subsections</h4><p>Each row owns only its text before the next heading. "
                 "Use Apply to subsections to assign a parent label to all children; individual labels remain editable. "
                 "Use Not categorizable for parts that belong in the counts but do not fit a specific category. Abstracts, references and other unwanted parts can be marked Exclude from summary; their text is retained.</p>"),
    paper_include_headings,
    widgets.HTML("<p>Heading words are included by default, matching the original word-count convention. Paragraph counts always exclude headings. "
                 "Suggestions and heading boundaries are heuristics; review them against the paper.</p>"),
    paper_suggest_button, paper_section_rows, paper_preview_panel,
    widgets.HTML("<h4>Category summary</h4><p>Copy titles (one per line) or plain-digit counts from each cell, or copy the whole table for Excel. "
                 "Section titles provide parent context for assigned subsections; word and paragraph totals count each part once. "
                 "The cells and buttons stay fixed. Long title lists scroll inside their cells; on a narrow screen, scroll the whole table horizontally.</p>"),
    paper_summary_css, paper_completion_warning, paper_summary_box, paper_totals, paper_copy_table_button, paper_clipboard_status, paper_copy_fallback,
    widgets.HTML("<h4>Save and reload</h4><p>Save creates a UTF-8 cleaned-text file and a self-contained JSON with text, section boundaries, "
                 "labels and counting settings. Reload the JSON here to revise labels later. In project mode, every save uses the project's "
                 "paper_categorizations subfolder, including when loading older files from elsewhere. Files are saved on the computer running the notebook.</p>"),
    widgets.HTML("<h4>Project papers and saved categorizations</h4><p>Projects made by the project notebook are detected from project.json. "
                 "Choose a sampled paper to start/reopen it, or associate text already on screen with the selected paper. "
                 "Saved JSON lists categorization files anywhere inside the project; metadata exports are not shown.</p>"),
    paper_project_folder, paper_scan_project_button, paper_project_status,
    paper_sample_choice, widgets.HBox([paper_use_sample_button, paper_associate_button]), paper_active_info,
    paper_saved_choice, paper_load_saved_button, paper_replace_current,
    paper_name, paper_folder, widgets.HBox([paper_save_button, paper_overwrite]), paper_save_status,
    paper_upload, paper_json_path, paper_load_path_button,
])
workflow_tabs = widgets.Tab(children=[legacy_workflow, paper_workflow])
workflow_tabs.set_title(0, "Clean & count (original)")
workflow_tabs.set_title(1, "Categorize full paper")
display(workflow_tabs)
